## naver_news 테이블의 한 행을 표현하기 위한 클래스 생성

In [14]:
# 네이버 뉴스 응답 데이터를 담을 NaverNews 엔티티 클래스 정의
from datetime import datetime

class NaverNews:

    def __init__(self, id: int, title: str, originallink: str, link: str, description: str, pub_date: str, created_at: datetime = None):
        self.__id = id
        self.__title = title
        self.__originallink = originallink
        self.__link = link
        self.__description = description
        self.__pub_date = pub_date
        self.__created_at = created_at

    @property
    def id(self):
        return self.__id

    @property
    def title(self):
        return self.__title

    # __title 속성에 대한 setter 메소드 예시
    # 아래 주석을 해제하면 naver_news.title = "새 제목"처럼 값을 변경할 수 있다.
    # @title.setter
    # def title(self, value):
    #     self.__title = value

    @property
    def originallink(self):
        return self.__originallink

    @property
    def link(self):
        return self.__link

    @property
    def description(self):
        return self.__description

    @property
    def pub_date(self):
        return self.__pub_date

    @property
    def created_at(self):
        return self.__created_at

    # naver new api 응답 데이터를
    # NaverNews 객체로 변환하기 위한 클래스 메서드
    @classmethod
    def from_api_item(cls, item:dict):
        return cls(
            id=None,
            title=item.get('title'),
            originallink=item.get('originallink'),
            link=item.get('link'),
            description=item.get('description'),
            pub_date=item.get('pubDate'),
        )


    def __repr__(self):
        # 객체를 print() 했을 때 어떤 값이 들어 있는지 확인하기 위한 문자열 표현 메소드
        return f'NaverNews({self.__id}, {self.__title}, {self.__originallink}, {self.__link}, {self.__description}, {self.__pub_date}, {self.__created_at})'

## .env를 읽어서 환경변수로 등록

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

NAVER_CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
NAVER_CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")

headers = {
    "X-Naver-Client-Id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
}

config = {
    "host": os.getenv("DB_HOST", "localhost"), # (key, default)
    "port": os.getenv("DB_PORT", "3306"),  # 기본포트 3306인 경우 생략가능
    "user": os.getenv("DB_USER", "skn_ai"),
    "password": os.getenv("DB_PASSWORD", "1234"),
    "database": os.getenv("DB_DATABASE", "menudb")
}

if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET:
    raise ValueError('NAVER_CLIENT_ID 또는 NAVER_CLIENT_SECRET이 설정되지 않았습니다.')

## 네이버 뉴스 "인공지능" 관련 최근 뉴스 10개 조회해서 DB에 저장

In [6]:
import requests
from pprint import pprint # 출력 시 공백문자를 이용해 가독성 좋게 출력

url = 'https://openapi.naver.com/v1/search/news.json'

params = {
    'query' : '인공지능',
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

# 수집한 news 데이터를 저장할 리스트
naver_news_list: list[NaverNews] = []

try:
    # GET Method == 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=params, # dict -> 쿼리 스트링 변환(+url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대 인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code # 상태 코드
    data = response.json() # 응답 데이터(json) -> dict 변환

    print(data)


    print('response_code: ', response_code)


except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")
except ValueError:
    print("응답 데이터가 JSON 형식이 아닙니다")

{'display': 10,
 'items': [{'description': '회담 직후 국세청은 홈택스 기반 전자신고·납부 체계부터 <b>인공지능</b>(AI) 챗봇 '
                           '상담 서비스에 이르기까지 ‘한국 국세청의 디지털 전환’ 사례를 소개했다. 가나 측은 최근 '
                           '디지털 기반 세정 현대화를 핵심 정책 과제로... ',
            'link': 'https://n.news.naver.com/mnews/article/666/0000111289?sid=100',
            'originallink': 'https://www.kyeonggi.com/article/20260615580534',
            'pubDate': 'Mon, 15 Jun 2026 16:19:00 +0900',
            'title': '국세청, 아프리카 가나와 징수공조 협력…재산 은닉 차단'},
           {'description': '안 장관은 “인구절벽과 AI(<b>인공지능</b>)·무인체계 등 유례없는 안보환경 변화에 '
                           '대응하기 위해, 미래 ‘정예 장교 양성’을 위한 사관학교 경쟁력 강화는 선택이 아닌 '
                           '필수과제”라고 강조했다. 이어 “첨단 교육 시설 및... ',
            'link': 'https://n.news.naver.com/mnews/article/021/0002797833?sid=100',
            'originallink': 'https://www.munhwa.com/article/11595658?ref=naver',
            'pubDate': 'Mon, 15 Jun 2026 16:18:00 +0900',
            'title': '안규백 “명품사관학교 창설”… 국군사관

In [16]:
import requests
from pprint import pprint # 출력 시 공백문자를 이용해 가독성 좋게 출력

url = 'https://openapi.naver.com/v1/search/news.json'

params = {
    'query' : '인공지능',
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

# 수집한 news 데이터를 저장할 리스트
naver_news_list:list[NaverNews] = []

try:
    # GET Method == 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=params, # dict -> 쿼리 스트링 변환(+url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대 인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code # 상태 코드
    data = response.json() # 응답 데이터(json) -> dict 변환

    # pprint(data)
    # 응답 데이터 중 뉴스 기사 리스트인 'items' 사용
    items = data.get('items', []) # (key, default)

    naver_news_list = [NaverNews.from_api_item(item) for item in items]

    print('response_code: ', response_code)
    print('naver_news_list: ', naver_news_list)


except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")
except ValueError:
    print("응답 데이터가 JSON 형식이 아닙니다")

response_code:  200
naver_news_list:  [NaverNews(None, 노신정 법무법인 대륙아주 변호사, ‘AI 기본법이 뭐길래’ 발간, https://www.sedaily.com/article/20055944?ref=naver, https://n.news.naver.com/mnews/article/011/0004631301?sid=102, 법무법인 대륙아주는 노신정 변호사가 <b>인공지능</b>기본법 안내서인 ‘AI기본법이 뭐길래’를 발간했다고 15일 밝혔다. 대륙아주에 따르면 이 책은 독자가 <b>인공지능</b>기본법 관련한 자신의 상황에 맞춰 법을 이해하고 적용할 수... , Mon, 15 Jun 2026 16:37:00 +0900, None), NaverNews(None, 심홍순 경기도의원 예산 수요예측, 사업설계의 총체적 재점검 필요, https://www.seoul.co.kr/news/publicnews/local_govern/kyungki_do/2026/06/15/20260615500197?wlog_tag3=naver, https://n.news.naver.com/mnews/article/081/0003652659?sid=102, 그는 “AI 행정혁신 서비스 시범 운영, <b>인공지능</b> 학습 데이터 1만 5390건 수집 등 초기 실적 자체는 긍정적”이라면서도 “중단 없이 준공까지 이어지는 것이 핵심인 만큼 다음 연도 상반기 내 지출 완료 일정을... , Mon, 15 Jun 2026 16:37:00 +0900, None), NaverNews(None, “코스닥 상장사 20%가 상폐 대상 위기” 벤처업계 李정부 자본시장 개..., https://biz.heraldcorp.com/article/10772083?ref=naver, https://n.news.naver.com/mnews/article/016/0002656842?sid=101, 특히 <b>인공지능</b>(AI), 바이오, 딥테크 기업들은 연구개발(R&amp;

In [17]:
import mysql.connector

try:
    with mysql.connector.connect(**config) as conn:
        with conn.cursor() as cursor:
            for naver_news in naver_news_list:
                cursor.execute('''
                insert into naver_news
                    (title, originallink, link, description, pub_date)
                    VALUES (%s, %s, %s, %s, %s)
                ''', (naver_news.title,
                       naver_news.originallink,
                       naver_news.link,
                       naver_news.description,
                       naver_news.pub_date
                   ))
            conn.commit() # 전체 insert 내용 커밋



except mysql.connector.Error as err:
    print("DB에러: ", err)
    conn.rollback() # 오류 발생 시 전체 rollback